# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset package using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. The entire workflow is based on referencing elements (record sets, fields, etc.) using their unique `@id` values, to ensure portability and reproducibility.

### Dataset Source
FAIR^2 dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

We first load the dataset schema and tabular data using `mlcroissant`. The dataset is defined by its Croissant JSON-LD file.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Number of record sets: {len(meta.record_sets)}")

## 2. Data Overview

Let's list all available record sets in the dataset, and for each, inspect their fields (columns), all referenced by their unique `@id`s. This provides a roadmap for later data extraction.

In [ ]:
# List all record sets and their fields by @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', 'No description')}")

    if hasattr(record_set, 'fields') and record_set.fields:
        print(f"  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'N/A')})")
    else:
        print("  (No fields defined)")

## 3. Data Extraction

We now extract records from each available record set. Each record set is referenced by its unique `@id`. The loaded data will be stored as a Pandas DataFrame for ease of analysis.

*Tip:* Use the `@id` values from above to access specific record sets and fields.

In [ ]:
# Prepare a list of all record set @ids
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    # Each record is a dict with field @ids as keys
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set @id: {rs_id}")

# Display available fields in the first record set as an example
if len(record_sets_ids) > 0:
    main_rs = record_sets_ids[0]
    print(f"Columns (@id) for record set {main_rs}:")
    print(list(dataframes[main_rs].columns))
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some basic data processing to the main record set (we'll use the first one listed, or select the most relevant one if there are multiple).

All operations use column references by their `@id`.

- Filter records based on a numeric value
- Normalize a numeric field
- Group and aggregate by a categorical field

In [ ]:
# For demonstration, select the 'Age' field if present (by @id), otherwise select any numeric field
import numpy as np
main_rs_id = record_sets_ids[0]
df = dataframes[main_rs_id]
numeric_field_id = None
group_field_id = None

# Try to find appropriate IDs
for rs in dataset.metadata.record_sets:
    if rs.id == main_rs_id:
        for field in rs.fields:
            if (getattr(field, 'data_type', '') or '').lower() in ['float', 'integer', 'number'] or 'age' in field.name.lower():
                numeric_field_id = field.id
            if 'sex' in field.name.lower() or 'gender' in field.name.lower() or 'msi' in field.name.lower():
                group_field_id = field.id

if numeric_field_id is None:
    # Fallback to the first column
    numeric_field_id = df.columns[0]

print(f"Using numeric field @id: {numeric_field_id}")
if group_field_id:
    print(f"Using group/categorical field @id: {group_field_id}")

# Try to cast the numeric column to float (skip NaN rows for stats)
def to_numeric(series):
    return pd.to_numeric(series, errors='coerce')
# Filter for values exceeding a threshold (as example: mean+1std, or >10)
numeric_series = to_numeric(df[numeric_field_id])
if numeric_series.dropna().empty:
    print('No numeric data available to filter/EAD')
else:
    threshold = numeric_series.mean() + numeric_series.std() if numeric_series.std() > 0 else numeric_series.mean()
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (to_numeric(filtered_df[numeric_field_id]) - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)

## 5. Visualization

Plot the distribution of the selected numeric field, and (if available) compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(to_numeric(df[numeric_field_id]), bins=12, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=to_numeric(df[numeric_field_id]))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we successfully:
- Loaded and inspected dataset metadata and structure from the Croissant schema.
- Extracted record sets and fields using precise `@id` references.
- Performed initial exploratory data analysis, including filtering and normalization of numeric fields, and aggregation by categorical fields.
- Visualized key data distributions.

**Tip:** For in-depth analysis, consult the official field descriptions and the [Croissant metadata schema](https://mlcommons.org/croissant/) for meanings of each `@id`.